In [1]:
!pip install pandas numpy faker

"pip" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.


In [2]:
import random
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

fake = Faker('es_CO')
Faker.seed(42)
np.random.seed(42)
random.seed(42)

def generate_synthetic_payments_dataset_v10(num_records=5000):
    users_pool = []
    for u_id in range(1001, 1501):
        users_pool.append({
            'user_id': f"USR-{u_id}",
            'user_avg_ticket_size': round(random.uniform(25000, 350000), -2),
            'user_cashback_responsiveness': round(random.uniform(0.10, 0.95), 2),
            'user_preferred_method': random.choice(['DIRECT_ACCOUNT_TRANSFER', 'CREDIT_CARD_BANK', 'DEBIT_CARD']),
            'user_credit_limit_utilization': round(random.uniform(0.05, 0.90), 2)
        })

    mcc_categories = [
        {'code': '5812', 'name': 'Restaurantes', 'avg_ticket': 45000},
        {'code': '5411', 'name': 'Supermercados', 'avg_ticket': 120000},
        {'code': '5541', 'name': 'Estaciones de Servicio', 'avg_ticket': 80000},
        {'code': '5651', 'name': 'Tiendas de Ropa', 'avg_ticket': 210000},
        {'code': '5912', 'name': 'Farmacias', 'avg_ticket': 35000}
    ]

    merchants_pool = []
    for m_id in range(501, 601):
        mcc_info = random.choice(mcc_categories)
        avg_t = round(mcc_info['avg_ticket'] * random.uniform(0.8, 1.3), -2)
        merchants_pool.append({
            'merchant_id': f"MERCH-{m_id}",
            'merchant_name': f"{fake.company()} - {mcc_info['name']}",
            'mcc_code': mcc_info['code'],
            'mcc_category': mcc_info['name'],
            'merchant_avg_ticket_size': avg_t,
            'merchant_accepts_installment': mcc_info['code'] in ['5651', '5411'],
            'merchant_tx_volume_30d': random.randint(150, 4500)
        })

    records = []
    start_date = datetime(2025, 1, 1)

    for i in range(num_records):
        user = random.choice(users_pool)
        merchant = random.choice(merchants_pool)

        random_days = random.randint(0, 180)
        random_hours = random.randint(7, 22)
        tx_timestamp = start_date + timedelta(days=random_days, hours=random_hours, minutes=random.randint(0, 59))

        base_amount = merchant['merchant_avg_ticket_size']
        tx_amount = max(5000, round(np.random.normal(base_amount, base_amount * 0.35), -2))

        user_balance = max(10000, round(tx_amount * random.uniform(0.5, 8.0), -2))
        user_balance_account_ratio = round(user_balance / tx_amount, 2)

        tx_channel = random.choice(['QR_STATIC', 'QR_DYNAMIC', 'E_COMMERCE_GATEWAY', 'POS_PHYSICAL'])
        is_payroll_day = tx_timestamp.day in [14, 15, 29, 30, 31]

        incentive_active_flag = random.choice([True, False]) if merchant['mcc_category'] in ['Restaurantes', 'Tiendas de Ropa', 'Supermercados'] else False

        if incentive_active_flag:
            incentive_pct = random.choice([0.05, 0.10, 0.15, 0.20])
            incentive_value_cop = round(min(tx_amount * incentive_pct, 25000), -2)
        else:
            incentive_pct = 0.0
            incentive_value_cop = 0.0

        pref_code = 0 if user['user_preferred_method'] == 'CREDIT_CARD_BANK' else (2 if user['user_preferred_method'] == 'DIRECT_ACCOUNT_TRANSFER' else 1)

        # Ajuste de constantes (+0.8 a Crédito y -0.8 a Débito) para igualar el soporte en test (~333 datos cada una)
        z_credit = (
            3.0 * (tx_amount / 180000.0) -
            2.5 * user['user_credit_limit_utilization'] +
            2.0 * (1.0 if pref_code == 0 else 0.0) +
            0.8 * (1.0 if merchant['merchant_accepts_installment'] else 0.0) + 0.8
        )

        z_transfer = (
            2.8 * (incentive_value_cop / 12000.0) * user['user_cashback_responsiveness'] +
            2.5 * (user_balance_account_ratio / 2.5) +
            2.0 * (1.0 if pref_code == 2 else 0.0) - 0.2
        )

        z_debit = (
            2.5 * (user_balance_account_ratio / 2.5) +
            2.0 * (1.0 if pref_code == 1 else 0.0) +
            1.5 * (1.0 - min(tx_amount / 200000.0, 1.0)) - 0.8
        )

        exp_c = np.exp(z_credit)
        exp_t = np.exp(z_transfer)
        exp_d = np.exp(z_debit)
        total_exp = exp_c + exp_t + exp_d

        p_credit = exp_c / total_exp
        p_transfer = exp_t / total_exp
        p_debit = exp_d / total_exp

        if random.random() < 0.04:
            chosen_payment_method = random.choice(['CREDIT_CARD_BANK', 'DEBIT_CARD', 'DIRECT_ACCOUNT_TRANSFER'])
        else:
            chosen_payment_method = np.random.choice(
                ['CREDIT_CARD_BANK', 'DIRECT_ACCOUNT_TRANSFER', 'DEBIT_CARD'],
                p=[p_credit, p_transfer, p_debit]
            )

        records.append({
            'tx_id': f"TX-{100000 + i}",
            'timestamp': tx_timestamp.strftime('%Y-%m-%d %H:%M:%S'),
            'user_id': user['user_id'],
            'user_avg_ticket_size': user['user_avg_ticket_size'],
            'user_cashback_responsiveness': user['user_cashback_responsiveness'],
            'user_credit_limit_utilization': user['user_credit_limit_utilization'],
            'user_balance_account_ratio': user_balance_account_ratio,
            'merchant_id': merchant['merchant_id'],
            'merchant_name': merchant['merchant_name'],
            'mcc_code': merchant['mcc_code'],
            'mcc_category': merchant['mcc_category'],
            'merchant_avg_ticket_size': merchant['merchant_avg_ticket_size'],
            'merchant_accepts_installment': merchant['merchant_accepts_installment'],
            'merchant_tx_volume_30d': merchant['merchant_tx_volume_30d'],
            'tx_amount': tx_amount,
            'tx_hour': tx_timestamp.hour,
            'tx_day_of_week': tx_timestamp.weekday(),
            'tx_channel': tx_channel,
            'is_payroll_day': is_payroll_day,
            'incentive_active_flag': incentive_active_flag,
            'incentive_pct': incentive_pct,
            'incentive_value_cop': incentive_value_cop,
            'target_chosen_payment_method': chosen_payment_method
        })

    return pd.DataFrame(records)

if __name__ == "__main__":
    df_payments = generate_synthetic_payments_dataset_v10(num_records=5000)
    df_payments.to_csv("dataset_medios_pago_sintetico.csv", index=False)
    print("✅ Archivo CSV v10 rebalanceado generado correctamente.")

✅ Archivo CSV v10 rebalanceado generado correctamente.
